In [58]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("cdaclab/unicon")

print("Path to dataset files:", path)


Path to dataset files: C:\Users\aiszk\.cache\kagglehub\datasets\cdaclab\unicon\versions\1


In [59]:
df_weather = pd.read_csv(f"{path}\weather_data.csv")
df_bc = pd.read_csv(f"{path}/building_consumption.csv")
df_events = pd.read_csv(f"{path}/events.csv")
df_cal = pd.read_csv(f"{path}/calender.csv")
df_building_meta = pd.read_csv(f"{path}/building_meta.csv")
df_campus_meta = pd.read_csv(f"{path}/campus_meta.csv")


In [60]:
datasets = [df_weather, df_bc, df_events, df_cal, df_building_meta, df_campus_meta]

In [66]:
for h in datasets:
    print(h.info(), h.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7396520 entries, 0 to 7396519
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   campus_id              int64  
 1   timestamp              object 
 2   apparent_temperature   float64
 3   air_temperature        float64
 4   dew_point_temperature  float64
 5   relative_humidity      float64
 6   wind_speed             float64
 7   wind_direction         float64
dtypes: float64(6), int64(1), object(1)
memory usage: 451.4+ MB
None           campus_id  apparent_temperature  air_temperature  \
count  7.396520e+06          7.396520e+06     7.396520e+06   
mean   2.480264e+00          1.316374e+01     1.579196e+01   
std    1.107213e+00          7.931828e+00     7.859730e+00   
min    1.000000e+00         -9.800000e+00    -3.100000e+00   
25%    2.000000e+00          7.300000e+00     1.020000e+01   
50%    2.000000e+00          1.210000e+01     1.470000e+01   
75%    3.000000e+00      

In [53]:
df_events.columns

Index(['meter_id', 'event_type', 'date', 'event_description'], dtype='object')

In [52]:
df_bc['meter_id']

0           1
1           1
2           1
3           1
4           1
           ..
8095519    64
8095520    64
8095521    64
8095522    64
8095523    64
Name: meter_id, Length: 8095524, dtype: int64

In [72]:
datasets = {
    'df_weather': df_weather,
    'df_bc': df_bc,
    'df_events': df_events,
    'df_cal': df_cal,
    'df_building_meta': df_building_meta,
    'df_campus_meta': df_campus_meta
}

for name, df in datasets.items():
    print(f"dla datasetu {name}: {list(df.columns)}")


dla datasetu df_weather: ['campus_id', 'timestamp', 'apparent_temperature', 'air_temperature', 'dew_point_temperature', 'relative_humidity', 'wind_speed', 'wind_direction']
dla datasetu df_bc: ['campus_id', 'meter_id', 'timestamp', 'consumption']
dla datasetu df_events: ['meter_id', 'event_type', 'date', 'event_description']
dla datasetu df_cal: ['date', 'is_holiday', 'is_semester', 'is_exam']
dla datasetu df_building_meta: ['campus_id', 'id', 'built_year', 'category', 'gross_floor_area', 'room_area', 'capacity']
dla datasetu df_campus_meta: ['id', 'name', 'capacity']


In [17]:
def preprocess_and_merge(df_weather, df_bc, df_events, df_cal):
    # Konwersja dat
    df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'])
    df_bc['timestamp'] = pd.to_datetime(df_bc['timestamp'])
    df_events['date'] = pd.to_datetime(df_events['date'])
    df_cal['date'] = pd.to_datetime(df_cal['date'])

    # Agregacja do poziomu dni
    df_bc['date'] = df_bc['timestamp'].dt.date
    df_weather['date'] = df_weather['timestamp'].dt.date

    # Grupowanie zużycia po dniach
    df_bc_daily = df_bc.groupby(['campus_id', 'meter_id', 'date'], as_index=False)['consumption'].sum()

    # Agregacja pogodowa po dniach i kampusach
    df_weather_daily = df_weather.groupby(['campus_id', 'date'], as_index=False).agg({
        'air_temperature':'mean',
        'apparent_temperature':'mean',
        'dew_point_temperature':'mean',
        'relative_humidity':'mean',
        'wind_speed':'mean',
        'wind_direction':'mean'
    })

    # Łączenie zużycia i pogody
    df = pd.merge(df_bc_daily, df_weather_daily, on=['campus_id','date'], how='left')

    df['date'] = pd.to_datetime(df['date'])
    df_cal['date'] = pd.to_datetime(df_cal['date'])

    # Dołączenie danych kalendarza
    df = pd.merge(df, df_cal, on='date', how='left')

    # Eventy: stworzenie cechy "days_since_last_event"
    df_events['date'] = pd.to_datetime(df_events['date'])
    df_events = df_events.sort_values(by=['meter_id', 'date'])

    last_event_date = df_events.groupby('meter_id')['date'].max().reset_index().rename(columns={'date':'last_event_date'})
    df = pd.merge(df, last_event_date, on='meter_id', how='left')

    df['days_since_last_event'] = (pd.to_datetime(df['date']) - df['last_event_date']).dt.days
    df['days_since_last_event'].fillna(9999, inplace=True) #wypełnia NaN, aby model rozróżnił przypadki "nigdy nie było eventu" vs. "event był niedawno",

    return df


In [21]:
df = preprocess_and_merge(df_weather, df_bc, df_events, df_cal)

In [22]:
df

,campus_id,meter_id,date,consumption,air_temperature,apparent_temperature,dew_point_temperature,relative_humidity,wind_speed,wind_direction,is_holiday,is_semester,is_exam,last_event_date,days_since_last_event
0,1,1,2019-03-29,33.54200,20.072917,15.899861,4.370486,40.567361,15.390069,170.987500,0.0,1,0,NaT,9999.0
1,1,1,2019-03-30,9.76000,10.625069,6.558819,6.292639,75.538194,16.562708,271.647222,0.0,1,0,NaT,9999.0
2,1,1,2019-03-31,1.93300,13.774722,11.831319,10.869931,83.525000,11.550903,251.790278,0.0,1,0,NaT,9999.0
3,1,1,2019-04-01,24.83400,14.107917,12.752292,8.414583,70.174306,5.217500,215.350000,0.0,1,0,NaT,9999.0
4,1,1,2019-04-02,18.05400,15.676875,13.617778,7.210903,64.432639,7.275069,161.152778,0.0,1,0,NaT,9999.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88788,3,53,2021-11-02,197.94057,20.662500,16.785417,7.327083,47.125000,NaN,NaN,0.0,0,0,NaT,9999.0
88789,3,53,2022-01-17,304.24120,24.877083,22.281250,10.541667,42.937500,NaN,NaN,0.0,0,0,NaT,9999.0
88790,3,53,2022-01-18,231.77353,19.164583,14.914583,8.522917,53.166667,NaN,NaN,0.0,0,0,NaT,9999.0
88791,3,53,2022-01-19,268.24535,22.350000,18.377083,9.706250,47.687500,NaN,NaN,0.0,0,0,NaT,9999.0


In [45]:
df[df['days_since_last_event'] < 0]





,campus_id,meter_id,date,consumption,air_temperature,apparent_temperature,dew_point_temperature,relative_humidity,wind_speed,wind_direction,is_holiday,is_semester,is_exam,last_event_date,days_since_last_event
2262,1,4,2018-01-01,307.3670,19.368125,18.536667,12.392014,67.898611,8.229722,172.374306,1.0,0,0,2021-07-01,-1277.0
2263,1,4,2018-01-02,538.6940,18.711736,16.538681,10.877708,61.620833,12.739097,195.447222,0.0,1,0,2021-07-01,-1276.0
2264,1,4,2018-01-03,581.9020,17.523264,15.476597,10.295417,66.134028,11.577917,211.365278,0.0,1,0,2021-07-01,-1275.0
2265,1,4,2018-01-04,649.0000,18.491944,17.361875,11.472500,66.411806,8.289097,178.658333,0.0,1,0,2021-07-01,-1274.0
2266,1,4,2018-01-05,582.7520,22.298056,21.963333,12.436319,60.009028,5.664444,191.767361,0.0,1,0,2021-07-01,-1273.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81736,2,14,2020-03-03,1573.8750,19.657778,18.104653,8.247986,49.758333,5.970278,152.365278,0.0,0,0,2020-03-08,-5.0
81737,2,14,2020-03-04,1780.5000,19.175694,19.967917,17.078194,88.902083,8.581250,117.377083,0.0,0,0,2020-03-08,-4.0
81738,2,14,2020-03-05,1795.6875,19.852361,20.119583,19.484375,97.809722,16.412222,47.916667,0.0,0,0,2020-03-08,-3.0
81739,2,14,2020-03-06,1563.4375,21.143403,20.619444,15.789236,74.570833,12.879931,269.739583,0.0,0,0,2020-03-08,-2.0


In [31]:
df['last_event_date'].value_counts()

2020-03-14    13660
2020-03-21    10590
2020-03-07     5772
2021-10-01     4726
2020-03-15     4602
2020-04-10     3596
2021-04-01     3152
2020-11-20     3149
2020-03-28     3144
2020-04-02     2633
2020-03-17     2631
2020-03-08     1677
2021-08-01     1576
2021-01-12     1576
2021-07-01     1576
2021-12-01     1576
2020-04-17     1576
2020-11-24     1576
2020-03-23     1575
2020-03-27     1575
2020-04-21     1571
2020-03-31     1401
2020-04-05     1134
2021-01-01     1056
2020-03-29     1056
Name: last_event_date, dtype: int64